# End-to-End Evaluation — results & trend

Reads the JSON reports written by `scripts/evaluate_all.py` and renders them as
tables and trend charts, so results can be compared **across evaluation sessions
over time** instead of scrolling terminal output.

The reports in `data/evaluations/` **are** the history — this notebook only reads
them. Re-running §1 is optional and only needed to add a new point; everything
below works offline from what is already on disk.

The unit of comparison is the **session**: one `evaluate_all.py` invocation runs
the pipeline over the real data *and* over the synthetic set and scores both, so
one session is one measurement of one state of the system even though it spans
two pipeline runs.

**Two things to keep in mind while reading (see `docs/End-to-End-Evaluation-Guide.md`):**

- **Leakage.** The Stage-4.25 gate and the Stage-4.5 matcher were both trained on
  the gold labels. Read `gate_pass` / `ml_auto_merge` only from a `strict`
  holdout report. The **clustering headline is leakage-free at any holdout**
  while `fs_feeds_clustering` and `ml_feeds_clustering` are both off, because
  clustering then uses only deterministic-rule edges.
- **Each stage is scored on the pool it actually saw.** The gate never sees pairs
  the rules already auto-merged; the matcher only sees gate survivors. The
  `scored_pairs` column says how many labeled pairs reached each stage.

## 0. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path


def _service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")


SERVICE_ROOT = _service_root(Path.cwd())
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import settings
from src.evaluation.report_io import (
    cluster_frame,
    funnel_frame,
    load_reports,
    loss_frame,
    metric_history,
    stage_frame,
    summary_frame,
    transitivity_frame,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

print("service root:", SERVICE_ROOT)
print("runs dir    :", settings.runs_dir)

### Chart styling

Three categorical hues from the validated palette, assigned to series in a fixed
sorted order so a series keeps its colour when other series appear or disappear.
Every chart is followed by its table view.

In [ ]:
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # blue, orange, aqua
INK, MUTED, GRID_C = "#0b0b0b", "#52514e", "#e6e5e1"


def color_map(series_names) -> dict:
    """Stable series -> colour. Keyed on the series name, never on its rank in
    the current chart, so filtering one series out never repaints the others."""
    return {name: SERIES_COLORS[i % len(SERIES_COLORS)]
            for i, name in enumerate(sorted(series_names))}


plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.edgecolor": GRID_C,
    "axes.labelcolor": MUTED,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID_C,
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})

## 1. (Optional) Run evaluations to add a new point

Skip this if you only want to read existing results.

`scripts/evaluate_all.py` does the whole thing in one session: pipeline over the
real data → score vs gold at **both** holdouts → pipeline over synthetic → score
vs entity truth. Gold gets both holdouts on purpose — `none` gives the tighter
clustering headline (8x more labeled positives), `strict` gives the honest
gate/matcher numbers.

Set `REUSE_REAL_RUN` to an existing `run_id` to skip re-running the real-data
pipeline, which is the slow part.

In [ ]:
RUN_EVALUATIONS = False    # flip to True to add a new session
SESSION_ID = None          # e.g. "gate_v2_baseline"; None -> UTC timestamp
REUSE_REAL_RUN = None      # an existing run_id, to skip the slow real-data run

if RUN_EVALUATIONS:
    cmd = [sys.executable, "scripts/evaluate_all.py"]
    if SESSION_ID:
        cmd += ["--session-id", SESSION_ID]
    if REUSE_REAL_RUN:
        cmd += ["--reuse-real-run", REUSE_REAL_RUN]
    print("$", " ".join(str(c) for c in cmd), "\n")
    # Streamed rather than captured: the real-data pipeline run takes minutes
    # and a silent cell that long is indistinguishable from a hung one.
    proc = subprocess.run(cmd, cwd=SERVICE_ROOT)
    if proc.returncode:
        raise RuntimeError("evaluate_all.py failed — see the output above.")
else:
    print("Skipped — reading stored reports only.")

## 2. What results exist

In [ ]:
reports = load_reports()
print(f"{len(reports)} stored report(s)\n")

summary = summary_frame(reports)
summary

## 3. One report in detail

`FOCUS = 0` is the most recently evaluated report. Change it to inspect another
row of the table above.

In [ ]:
FOCUS = 0

report = reports[FOCUS]
holdout = report["leakage"]["restriction"]
print(f"run {report['run_id']}  |  {report['label_source']}  |  holdout: {holdout}")
print(f"evaluated {report['evaluated_utc']}  |  pipeline git sha: {report['git_sha']}")
print(f"{report['universe']['labeled_pairs']} labeled pairs, "
      f"{report['universe']['positives']} positive")
if report["leakage"].get("note"):
    print("\n!", report["leakage"]["note"])

### 3.1 Per-stage decisions

`scored_pairs` < `labeled_pairs` means the stage only saw part of the labeled
set — the gate and the matcher sit downstream of filters, so this is expected,
not a defect. Their precision/recall are computed on that sub-pool only.

In [ ]:
stage_frame(report)

### 3.2 Funnel and loss attribution

`clustered` is **not** a subset of the row above it: transitivity can merge a
pair that never even blocked.

In [ ]:
funnel_frame(report)

In [ ]:
losses = loss_frame(report)
losses

In [ ]:
def plot_losses(losses: pd.DataFrame, title: str):
    """Magnitude by category -> horizontal bars, sorted. One series, so no
    legend; every bar is directly labelled instead of relying on the axis."""
    data = losses[losses["missed"] > 0]
    if data.empty:
        print("No missed true pairs to attribute.")
        return
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(data) + 1.4))
    y = range(len(data))
    ax.barh(list(y), data["missed"], color=SERIES_COLORS[0], height=0.62)
    ax.set_yticks(list(y), data["stage"])
    ax.invert_yaxis()
    ax.set_xlabel("true pairs lost")
    ax.set_title(title, color=INK, loc="left")
    ax.grid(axis="y", visible=False)
    span = data["missed"].max()
    for i, (n, pct) in enumerate(zip(data["missed"], data["share_pct"])):
        ax.text(n + span * 0.015, i, f"{n:,}  ({pct}%)", va="center",
                fontsize=8.5, color=MUTED)
    ax.set_xlim(0, span * 1.25)
    plt.tight_layout()
    plt.show()


plot_losses(losses, f"Where true pairs were lost — {report['label_source']} / {holdout}")

### 3.3 Transitivity and cluster-level metrics

Transitive-only merges were never scored by any classifier, so their false
positives are invisible to every per-stage metric above.

In [ ]:
transitivity_frame(report)

In [ ]:
# `closure_contradictions` > 0 means the truth partition asserts pairs the
# labeller marked non-matches -> read these as directional, prefer the headline.
cluster_frame([report]).T

## 4. Comparison over time

One point per **session**, ordered by when it was evaluated. Series are
`source / holdout` combinations, so gold and synthetic sit on the same axes
without ever being averaged together — and both halves of a session share an
x-position even though they are different pipeline runs.

In [ ]:
def plot_metric_history(reports, metrics=("precision", "recall", "f1"),
                        stage="clustering"):
    """Small multiples — one panel per metric, one shared y-axis scale.

    Small multiples rather than one panel with a second y-axis: precision,
    recall and F1 share a 0-1 scale, and a dual-axis chart would invite
    comparing quantities that are not comparable.
    """
    frames = {m: metric_history(reports, m, stage) for m in metrics}
    frames = {m: f for m, f in frames.items() if not f.empty}
    if not frames:
        print(f"No stored reports carry a '{stage}' {'/'.join(metrics)} value yet.")
        return

    order = (pd.concat(frames.values())
             .drop_duplicates("session_id")
             .sort_values("evaluated_utc")["session_id"].tolist())
    all_series = sorted({s for f in frames.values() for s in f["series"].unique()})
    colors = color_map(all_series)

    fig, axes = plt.subplots(1, len(frames), figsize=(4.1 * len(frames), 3.5),
                             sharey=True)
    axes = [axes] if len(frames) == 1 else list(axes)

    for ax, (metric, frame) in zip(axes, frames.items()):
        for name, grp in frame.groupby("series"):
            grp = grp.set_index("session_id").reindex(order).dropna(subset=["value"])
            xs = [order.index(r) for r in grp.index]
            ax.plot(xs, grp["value"], marker="o", markersize=5.5, linewidth=2,
                    color=colors[name], label=name, zorder=3)
            if len(all_series) <= 4 and len(xs):
                # Direct-label the last point, staggered by series index —
                # with a single run on the x-axis every series lands on the
                # same tick and un-staggered labels overprint each other.
                dy = 6 - 11 * all_series.index(name)
                ax.annotate(f"{grp['value'].iloc[-1]:.3f}",
                            (xs[-1], grp["value"].iloc[-1]),
                            textcoords="offset points", xytext=(8, dy),
                            fontsize=8, color=colors[name])
        ax.set_title(f"{stage} {metric}", color=INK, loc="left")
        ax.set_ylim(0, 1.05)
        ax.set_xticks(range(len(order)), order, rotation=35, ha="right", fontsize=7.5)
        ax.grid(axis="x", visible=False)

    handles, labels = axes[0].get_legend_handles_labels()
    if len(labels) >= 2:  # a single series is named by the title instead
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 3),
                   bbox_to_anchor=(0.5, 1.10), fontsize=8.5)
    fig.supxlabel("evaluation session (ordered by time)", color=MUTED, fontsize=8.5)
    plt.tight_layout()
    plt.show()


plot_metric_history(reports)

### 4.1 Table view of the same data

In [ ]:
history = pd.concat([metric_history(reports, m) for m in ("precision", "recall", "f1")],
                    ignore_index=True)
if history.empty:
    print("Nothing to compare yet — evaluate at least one run.")
else:
    display(history.pivot_table(index=["evaluated_utc", "session_id", "series"],
                                columns="metric", values="value").round(4))

### 4.2 Any stage, not just clustering

Swap `stage` for any row of the per-stage table — `rules_auto_merge`,
`gate_pass`, `ml_auto_merge`, `blocking`. Remember the leakage rule: only read
`gate_pass` / `ml_auto_merge` from `strict` reports.

In [ ]:
plot_metric_history(reports, metrics=("precision", "recall"), stage="rules_auto_merge")

## 5. Cluster-level metrics side by side

`truth_source` tells you how far to trust each row. Synthetic carries declared
entity ids, so its numbers are exact; gold's truth partition is inferred by
transitive closure of the positive pairs.

`nonsingleton_exact_rate` is the honest cluster number — the overall
`exact_rate` is dominated by the singletons the pipeline correctly leaves alone.

In [ ]:
cluster_frame(reports)